## 1. Preparación de Datos

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

In [ ]:
# Generar dataset sintético de gastos médicos
np.random.seed(42)
n_samples = 10000

# Generar variables
edad = np.random.randint(0, 70, n_samples)
sexo = np.random.choice(["M", "F"], n_samples)
imc = np.random.normal(28, 6, n_samples)
imc = np.clip(imc, 15, 50)  # Limitar valores razonables
fumador = np.random.choice([0, 1], n_samples, p=[0.8, 0.2]) & (edad > 18)
region = np.random.choice(["Norte", "Sur"], n_samples)

# Generar gastos médicos con relaciones no lineales
# La relación real incluye efectos no lineales:
# - Edad tiene efecto cuadrático (los gastos aumentan más rápido con la edad)
# - IMC tiene efecto exponencial para valores altos
# - Fumar multiplica los gastos significativamente
gastos_base = (
    100
    + (edad * 50)
    + ((edad - 20) ** 2 * 2)
    + np.random.uniform(size=n_samples) * (7500 / ((0.5 * edad + 1)))
)
gastos_imc = np.where(imc > 30, (imc - 30) ** 2 * 100, (imc - 20) * 50) / 10
gastos_fumador = (
    np.where(fumador == 1, np.random.uniform(size=n_samples) * (edad) ** 2.3, 0) / 2
)
gastos_maternidad = (
    np.where(
        (sexo == "F")
        & (edad < 46)
        & (edad > 16)
        & ((1 + np.abs(edad - 31)) * np.random.uniform(size=n_samples) < 1),
        np.maximum(0, -1 + np.abs((np.random.normal(size=n_samples) + 1)) ** 1.5),
        0,
    )
) * 2000
# Gastos totales con ruido
gastos = gastos_base + gastos_imc + gastos_fumador + gastos_maternidad
gastos = gastos + np.where(
    region == "Norte",
    np.random.normal(0, 1500, n_samples),
    np.random.normal(0, 1000, n_samples),
)
gastos = np.maximum(gastos, 100)

# Crear DataFrame
df = pd.DataFrame(
    {
        "edad": edad,
        "sexo": sexo,
        "imc": imc,
        "fumador": fumador,
        "region": region,
        "gastos": gastos,
    }
)

print(f"Dataset generado con {len(df)} registros")
df.head(10)

In [ ]:
# Exploración inicial de los datos
df.describe()

In [ ]:
# Información del dataset
df.info()

## 2. Análisis Exploratorio: Relaciones No Lineales

Antes de modelar, es importante visualizar las relaciones entre variables para identificar patrones no lineales.

In [ ]:
# Visualizar relación entre edad y gastos
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Relación edad-gastos por estado de fumador
for fumador_val, color, label in [(0, "blue", "No fumador"), (1, "red", "Fumador")]:
    mask = df["fumador"] == fumador_val
    axes[0].scatter(
        df[mask]["edad"], df[mask]["gastos"], alpha=0.5, s=30, c=color, label=label
    )
axes[0].set_xlabel("Edad", fontsize=12)
axes[0].set_ylabel("Gastos Médicos (€)", fontsize=12)
axes[0].set_title(
    "Relación Edad vs Gastos (por Fumador)", fontsize=14, fontweight="bold"
)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Relación IMC-gastos por estado de fumador
for fumador_val, color, label in [(0, "blue", "No fumador"), (1, "red", "Fumador")]:
    mask = df["fumador"] == fumador_val
    axes[1].scatter(
        df[mask]["imc"], df[mask]["gastos"], alpha=0.5, s=30, c=color, label=label
    )
axes[1].set_xlabel("IMC", fontsize=12)
axes[1].set_ylabel("Gastos Médicos (€)", fontsize=12)
axes[1].set_title(
    "Relación IMC vs Gastos (por Fumador)", fontsize=14, fontweight="bold"
)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualizar relación por sexo
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Obtener límites globales para mantener misma escala
edad_min, edad_max = df["edad"].min(), df["edad"].max()
gastos_min, gastos_max = df["gastos"].min(), df["gastos"].max()

# Relación edad-gastos para hombres (izquierda)
mask_m = df["sexo"] == "M"
axes[0].scatter(
    df[mask_m]["edad"], df[mask_m]["gastos"], alpha=0.5, s=30, c="steelblue"
)
axes[0].set_xlabel("Edad", fontsize=12)
axes[0].set_ylabel("Gastos Médicos (€)", fontsize=12)
axes[0].set_title("Hombres: Edad vs Gastos", fontsize=14, fontweight="bold")
axes[0].grid(alpha=0.3)
axes[0].set_xlim(edad_min, edad_max)
axes[0].set_ylim(gastos_min, gastos_max)

# Relación edad-gastos para mujeres (derecha)
mask_f = df["sexo"] == "F"
axes[1].scatter(df[mask_f]["edad"], df[mask_f]["gastos"], alpha=0.5, s=30, c="coral")
axes[1].set_xlabel("Edad", fontsize=12)
axes[1].set_ylabel("Gastos Médicos (€)", fontsize=12)
axes[1].set_title("Mujeres: Edad vs Gastos", fontsize=14, fontweight="bold")
axes[1].grid(alpha=0.3)
axes[1].set_xlim(edad_min, edad_max)
axes[1].set_ylim(gastos_min, gastos_max)

In [ ]:
# Visualización avanzada: Edad vs Gastos por Sexo
# Combinando violin plot, box plot y strip plot

sns.set_style("white")
palette = {"M": "steelblue", "F": "coral"}

fig = plt.figure(figsize=(12, 6))
ax = sns.violinplot(
    y="sexo", x="gastos", data=df, palette=palette, scale="width", inner=None
)

xlim = ax.get_xlim()
ylim = ax.get_ylim()

# Recortar violines por la mitad
for violin in ax.collections:
    bbox = violin.get_paths()[0].get_extents()
    x0, y0, width, height = bbox.bounds
    violin.set_clip_path(
        plt.Rectangle((x0, y0), width, height / 2, transform=ax.transData)
    )

# Agregar boxplot
sns.boxplot(
    y="sexo",
    x="gastos",
    data=df,
    saturation=1,
    width=0.3,
    boxprops={"zorder": 3, "facecolor": "none"},
    ax=ax,
)

# Agregar puntos (muestra)
old_len_collections = len(ax.collections)
# Tomar una muestra aleatoria para no saturar el gráfico
df_sample = df.sample(n=min(1000, len(df)), random_state=42)
sns.stripplot(
    y="sexo", x="gastos", data=df_sample, color="dodgerblue", alpha=0.4, size=3, ax=ax
)

for dots in ax.collections[old_len_collections:]:
    dots.set_offsets(dots.get_offsets() + np.array([0, 0.12]))

ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel("Gastos Médicos (€)", fontsize=12)
ax.set_ylabel("Sexo", fontsize=12)
ax.set_title("Distribución de Gastos Médicos por Sexo", fontsize=14, fontweight="bold")
ax.set_yticklabels(["Femenino", "Masculino"])

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de la variable objetivo
plt.figure(figsize=(8, 5))

plt.hist(df["gastos"], bins=50, edgecolor="black", alpha=0.7)
plt.xlabel("Gastos Médicos (€)", fontsize=12)
plt.ylabel("Frecuencia", fontsize=12)
plt.title("Distribución de Gastos Médicos", fontsize=14, fontweight="bold")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Modelo Baseline: Regresión Lineal Simple

Comenzamos con un modelo lineal simple (sin transformaciones) para establecer una línea base de rendimiento.

In [ ]:
# Preparar variables para modelado
# Convertir variables categóricas a numéricas
df_model = df.copy()
df_model["sexo_M"] = (df_model["sexo"] == "M").astype(int)
df_model = pd.get_dummies(df_model, columns=["region"], drop_first=True)

# Seleccionar variables base
feature_cols_base = ["edad", "sexo_M", "region_Sur", "imc", "fumador"]
X_base = df_model[feature_cols_base]
y = df_model["gastos"]

print(f"Variables: {feature_cols_base}")

In [ ]:
# Modelo lineal simple (sin dividir datos aún)
modelo_baseline = LinearRegression()
modelo_baseline.fit(X_base, y)

# Predicciones
y_pred_baseline = modelo_baseline.predict(X_base)

# Evaluación
baseline_r2 = r2_score(y, y_pred_baseline)
baseline_rmse = np.sqrt(mean_squared_error(y, y_pred_baseline))
baseline_mae = mean_absolute_error(y, y_pred_baseline)

print("=" * 60)
print("MODELO BASELINE: Regresión Lineal Simple")
print("=" * 60)
print(f"\nR² Score: {baseline_r2:.4f}")
print(f"RMSE (Error Cuadrático Medio): {baseline_rmse:.2f} €")
print(f"MAE (Error Absoluto Medio): {baseline_mae:.2f} €")

# Coeficientes
print(f"\nCoeficientes del modelo:")
for feature, coef in zip(feature_cols_base, modelo_baseline.coef_):
    print(f"   {feature:15s}: {coef:10.2f}")
print(f"   {'Intercepto':15s}: {modelo_baseline.intercept_:10.2f}")

In [ ]:
# Visualizar predicciones del modelo baseline
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Predicciones vs valores reales
axes[0].scatter(y, y_pred_baseline, alpha=0.5, s=30)
axes[0].plot(
    [y.min(), y.max()],
    [y.min(), y.max()],
    "r--",
    lw=2,
    label="Predicción perfecta",
)
axes[0].set_xlabel("Gastos Reales (€)", fontsize=12)
axes[0].set_ylabel("Gastos Predichos (€)", fontsize=12)
axes[0].set_title(
    "Modelo Baseline: Predicciones vs Reales", fontsize=14, fontweight="bold"
)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Residuos
residuos = y - y_pred_baseline
axes[1].scatter(y_pred_baseline, residuos, alpha=0.5, s=30)
axes[1].axhline(y=0, color="r", linestyle="--", lw=2)
axes[1].set_xlabel("Gastos Predichos (€)", fontsize=12)
axes[1].set_ylabel("Residuos (€)", fontsize=12)
axes[1].set_title("Análisis de Residuos", fontsize=14, fontweight="bold")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualizar predicciones del modelo baseline
plt.figure(figsize=(10, 6))

# Datos reales
plt.scatter(
    df["edad"], df["gastos"], alpha=0.3, s=20, color="black", label="Datos reales"
)

# Predicciones ordenadas por edad
idx = np.argsort(df["edad"])
edad_sorted = df["edad"].iloc[idx]
pred_sorted = modelo_baseline.predict(X_base.iloc[idx])

plt.scatter(
    edad_sorted,
    pred_sorted,
    color="red",
    s=10,
    label="Predicciones modelo baseline",
)

plt.xlabel("Edad", fontsize=12)
plt.ylabel("Gastos Médicos (€)", fontsize=12)
plt.title("Modelo Baseline: Edad vs Gastos Predichos", fontsize=14, fontweight="bold")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Transformaciones No Lineales para Mejorar el Baseline

Ahora aplicaremos diferentes transformaciones para capturar las relaciones no lineales que el modelo baseline no pudo capturar.

### 4.1 Transformación Logarítmica

Las transformaciones logarítmicas son útiles cuando:
- La variable tiene una distribución sesgada
- Las relaciones son multiplicativas en lugar de aditivas
- Queremos reducir el impacto de valores extremos

In [ ]:
# Aplicar transformación logarítmica a variables continuas
X_log = X_base.copy()

# Log de edad (sumamos 1 para evitar log(0))
X_log["edad_log"] = np.log1p(X_log["edad"])

# Log de IMC
X_log["imc_log"] = np.log1p(X_log["imc"])

# Entrenar modelo con transformaciones logarítmicas
modelo_log = LinearRegression()
modelo_log.fit(X_log, y)

y_pred_log = modelo_log.predict(X_log)

# Evaluación
log_r2 = r2_score(y, y_pred_log)
log_rmse = np.sqrt(mean_squared_error(y, y_pred_log))
log_mae = mean_absolute_error(y, y_pred_log)

print("=" * 60)
print("MODELO CON TRANSFORMACIÓN LOGARÍTMICA")
print("=" * 60)
print(f"\nR² Score: {log_r2:.4f}")
print(f"RMSE: {log_rmse:.2f} €")
print(f"MAE: {log_mae:.2f} €")
print(f"\nMejora respecto a baseline:")
print(f"   R² Score: {log_r2 - baseline_r2:+.4f}")
print(f"   RMSE: {log_rmse - baseline_rmse:+.2f} €")

print(f"\nNuevas variables: edad_log, imc_log")

In [ ]:
# Visualizar predicciones del modelo logarítmico
plt.figure(figsize=(10, 6))

# Datos reales
plt.scatter(
    df["edad"], df["gastos"], alpha=0.3, s=20, color="black", label="Datos reales"
)

# Predicciones ordenadas por edad
idx = np.argsort(df["edad"])
edad_sorted = df["edad"].iloc[idx]
pred_sorted = modelo_log.predict(X_log.iloc[idx])

plt.scatter(
    edad_sorted,
    pred_sorted,
    color="red",
    s=10,
    label="Predicciones modelo logarítmico",
)

plt.xlabel("Edad", fontsize=12)
plt.ylabel("Gastos Médicos (€)", fontsize=12)
plt.title(
    "Modelo Logarítmico: Edad vs Gastos Predichos", fontsize=14, fontweight="bold"
)
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 4.2 Transformación Polinómica

Las transformaciones polinómicas permiten capturar relaciones no lineales mediante la creación de nuevas variables que son potencias de las originales o multiplicaciones entre ellas.

**Ejemplo**: Para una variable $x$, podemos crear $x^2, x^3$, etc.

También se crean **términos de interacción**: $x_1 \times x_2$

In [ ]:
# Probar diferentes grados polinómicos
resultados_poly = []

for grado in [1, 2]:
    # Crear variables polinómicas
    poly = PolynomialFeatures(degree=grado, include_bias=False)
    X_poly = poly.fit_transform(X_base)

    # Entrenar modelo
    modelo_poly = LinearRegression()
    modelo_poly.fit(X_poly, y)

    # Predicciones
    y_pred_poly = modelo_poly.predict(X_poly)

    # Métricas
    r2_poly = r2_score(y, y_pred_poly)
    rmse_poly = np.sqrt(mean_squared_error(y, y_pred_poly))

    resultados_poly.append(
        {
            "grado": grado,
            "n_features": X_poly.shape[1],
            "r2": r2_poly,
            "rmse": rmse_poly,
        }
    )

# Mostrar resultados
df_poly = pd.DataFrame(resultados_poly)
print("=" * 70)
print("COMPARACIÓN DE MODELOS POLINÓMICOS (sin dividir datos)")
print("=" * 70)
print(df_poly.to_string(index=False))

In [ ]:
# Visualizar predicciones del modelo polinómico (grado 2)
plt.figure(figsize=(10, 6))

# Datos reales
plt.scatter(
    df["edad"], df["gastos"], alpha=0.3, s=20, color="black", label="Datos reales"
)

# Predicciones ordenadas por edad
idx = np.argsort(df["edad"])
edad_sorted = df["edad"].iloc[idx]
X_poly_sorted = poly.transform(X_base.iloc[idx])
pred_sorted = modelo_poly.predict(X_poly_sorted)

plt.scatter(
    edad_sorted,
    pred_sorted,
    color="red",
    s=10,
    label="Predicciones modelo polinómico (grado 2)",
)

plt.xlabel("Edad", fontsize=12)
plt.ylabel("Gastos Médicos (€)", fontsize=12)
plt.title(
    "Modelo Polinómico (Grado 2): Edad vs Gastos Predichos",
    fontsize=14,
    fontweight="bold",
)
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 4.3 Discretización de Variables Continuas

La discretización convierte variables continuas en categorías (bins). Esto puede ser útil para:
- Capturar efectos no lineales mediante "saltos" en categorías
- Simplificar interpretación (ej: "joven", "adulto", "mayor")
- Reducir impacto de outliers

**Desventaja**: Pérdida de información al convertir valores continuos en categorías. Se asume que todos los valores dentro de un bin son equivalentes.

In [ ]:
# Discretizar edad en grupos
X_disc = X_base.copy()

# Crear bins de edad
edad_bins = [-1, 30, 45, 60, 100]
edad_labels = ["0-29", "30-44", "45-59", "60+"]

X_disc["grupo_edad"] = pd.cut(X_disc["edad"], bins=edad_bins, labels=False)

# Crear bins de IMC
imc_bins = [0, 18.5, 25, 30, 100]
imc_labels = ["Bajo peso", "Normal", "Sobrepeso", "Obesidad"]

X_disc["grupo_imc"] = pd.cut(X_disc["imc"], bins=imc_bins, labels=False)

# Eliminar variables originales continuas
X_disc = X_disc.drop(["edad", "imc"], axis=1)

# Entrenar modelo con variables discretizadas
modelo_disc = LinearRegression()
modelo_disc.fit(X_disc, y)

y_pred_disc = modelo_disc.predict(X_disc)

# Evaluación
disc_r2 = r2_score(y, y_pred_disc)
disc_rmse = np.sqrt(mean_squared_error(y, y_pred_disc))

print("=" * 60)
print("MODELO CON DISCRETIZACIÓN")
print("=" * 60)
print(f"\nR² Score: {disc_r2:.4f}")
print(f"RMSE: {disc_rmse:.2f} €")
print(f"\nGrupos de edad: {edad_labels}")
print(f"Grupos de IMC:  {imc_labels}")
print(f"\nLa discretización simplifica el modelo pero puede perder información")

In [ ]:
# Visualizar predicciones del modelo con discretización
plt.figure(figsize=(10, 6))

# Datos reales
plt.scatter(
    df["edad"], df["gastos"], alpha=0.3, s=20, color="black", label="Datos reales"
)

# Predicciones ordenadas por edad
idx = np.argsort(df["edad"])
edad_sorted = df["edad"].iloc[idx]
pred_sorted = modelo_disc.predict(X_disc.iloc[idx])

plt.scatter(
    edad_sorted,
    pred_sorted,
    color="red",
    s=10,
    label="Predicciones modelo discretizado",
)

plt.xlabel("Edad", fontsize=12)
plt.ylabel("Gastos Médicos (€)", fontsize=12)
plt.title(
    "Modelo con Discretización: Edad vs Gastos Predichos",
    fontsize=14,
    fontweight="bold",
)
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. División Train/Test: Evaluar Generalización y Overfitting

Hasta ahora hemos entrenado y evaluado todos los modelos en los **mismos datos**. Esto no nos dice si el modelo generaliza bien a datos nuevos.

**¿Por qué dividir los datos?**

Para evaluar correctamente nuestros modelos, necesitamos:
- **Conjunto de entrenamiento**: datos usados para ajustar el modelo
- **Conjunto de validación (test)**: datos NO vistos durante el entrenamiento, usados para evaluar la capacidad de generalización

Esto nos permite detectar:
- **Overfitting**: el modelo memoriza los datos de entrenamiento pero falla con datos nuevos
- **Underfitting**: el modelo es demasiado simple y no captura los patrones de los datos

In [ ]:
# División 80-20: 80% entrenamiento, 20% validación
X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42
)

print(f"División de datos:")
print(
    f"   Entrenamiento: {len(X_train)} muestras ({len(X_train)/len(X_base)*100:.1f}%)"
)
print(f"   Validación:    {len(X_test)} muestras ({len(X_test)/len(X_base)*100:.1f}%)")
print(f"\nLos datos están listos para evaluar generalización")

### 5.1 Re-evaluar Modelos con Train/Test Split

In [ ]:
# 1. Modelo Baseline con train/test
modelo_baseline_tt = LinearRegression()
modelo_baseline_tt.fit(X_train, y_train)

y_train_pred = modelo_baseline_tt.predict(X_train)
y_test_pred = modelo_baseline_tt.predict(X_test)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print("=" * 60)
print("MODELO BASELINE con Train/Test Split")
print("=" * 60)
print(f"\nR² Score:")
print(f"   Entrenamiento: {train_r2:.4f}")
print(f"   Validación:    {test_r2:.4f}")
print(f"   Gap:           {train_r2 - test_r2:.4f}")
print(f"\nRMSE:")
print(f"   Entrenamiento: {train_rmse:.2f} €")
print(f"   Validación:    {test_rmse:.2f} €")

### 5.2 Modelos Polinómicos: Detectando Overfitting

In [ ]:
# 2. Modelo Logarítmico con train/test
X_train_log = X_train.copy()
X_test_log = X_test.copy()

X_train_log["edad_log"] = np.log1p(X_train_log["edad"])
X_train_log["imc_log"] = np.log1p(X_train_log["imc"])
X_test_log["edad_log"] = np.log1p(X_test_log["edad"])
X_test_log["imc_log"] = np.log1p(X_test_log["imc"])

modelo_log_tt = LinearRegression()
modelo_log_tt.fit(X_train_log, y_train)

y_train_pred_log = modelo_log_tt.predict(X_train_log)
y_test_pred_log = modelo_log_tt.predict(X_test_log)

train_r2_log = r2_score(y_train, y_train_pred_log)
test_r2_log = r2_score(y_test, y_test_pred_log)
train_rmse_log = np.sqrt(mean_squared_error(y_train, y_train_pred_log))
test_rmse_log = np.sqrt(mean_squared_error(y_test, y_test_pred_log))

print("=" * 60)
print("MODELO LOGARÍTMICO con Train/Test Split")
print("=" * 60)
print(f"\nR² Score:")
print(f"   Entrenamiento: {train_r2_log:.4f}")
print(f"   Validación:    {test_r2_log:.4f}")
print(f"   Gap:           {train_r2_log - test_r2_log:.4f}")
print(f"\nRMSE:")
print(f"   Entrenamiento: {train_rmse_log:.2f} €")
print(f"   Validación:    {test_rmse_log:.2f} €")

In [ ]:
# 3. polinómico
resultados_poly = []

X_train_poly = X_train.copy()
X_test_poly = X_test.copy()
print("=" * 60)
print("MODELO POLINÓMICO con Train/Test Split")
for grado in [1, 2, 3]:
    # Crear variables polinómicas
    poly = PolynomialFeatures(degree=grado, include_bias=False)
    X_train_poly_temp = poly.fit_transform(X_train)
    X_test_poly_temp = poly.transform(X_test)

    # Entrenar modelo
    modelo_poly_tt = LinearRegression()
    modelo_poly_tt.fit(X_train_poly_temp, y_train)

    y_train_pred_poly = modelo_poly_tt.predict(X_train_poly_temp)
    y_test_pred_poly = modelo_poly_tt.predict(X_test_poly_temp)

    train_r2_poly = r2_score(y_train, y_train_pred_poly)
    test_r2_poly = r2_score(y_test, y_test_pred_poly)
    train_rmse_poly = np.sqrt(mean_squared_error(y_train, y_train_pred_poly))
    test_rmse_poly = np.sqrt(mean_squared_error(y_test, y_test_pred_poly))

    print("=" * 60)
    print(f"\nGrado: {grado}")
    print("=" * 60)
    print(f"\nR² Score:")
    print(f"   Entrenamiento: {train_r2_poly:.4f}")
    print(f"   Validación:    {test_r2_poly:.4f}")
    print(f"   Gap:           {train_r2_poly - test_r2_poly:.4f}")
    print(f"\nRMSE:")
    print(f"   Entrenamiento: {train_rmse_poly:.2f} €")
    print(f"   Validación:    {test_rmse_poly:.2f} €")

In [ ]:
# Visualizar predicciones del modelo polinómico (grado 3)
plt.figure(figsize=(10, 6))

# Datos reales
plt.scatter(
    df["edad"], df["gastos"], alpha=0.3, s=20, color="black", label="Datos reales"
)

# Predicciones ordenadas por edad
idx = np.argsort(df["edad"])
edad_sorted = df["edad"].iloc[idx]

# Transformar todo el dataset con el polinomio de grado 3
poly_full = PolynomialFeatures(degree=3, include_bias=False)
X_full_poly = poly_full.fit_transform(X_base)
pred_sorted = modelo_poly_tt.predict(X_full_poly[idx])

plt.scatter(
    edad_sorted,
    pred_sorted,
    color="red",
    s=10,
    label="Predicciones modelo polinómico (grado 3)",
)

plt.xlabel("Edad", fontsize=12)
plt.ylabel("Gastos Médicos (€)", fontsize=12)
plt.title(
    "Modelo Polinómico (Grado 3): Edad vs Gastos Predichos",
    fontsize=14,
    fontweight="bold",
)
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()